In [1]:
import os
import copy
import time
import pickle
import numpy as np
import random
import argparse
from tqdm import tqdm
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import json
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from tqdm import tqdm
from tensorboardX import SummaryWriter
from pathlib import Path
from datetime import datetime
from utils import average_weights, exp_details, load_params
from defense import krum, multi_krum, detect_anomalies_by_distance, bulyan, detect_outliers_from_weights, trimmed_mean, detect_outliers_with_silhouette
from defense_utils import extract_lora_qs, extract_lora_vals, compute_wa_distances, compute_weighted_distance_with_attention

/Users/vblack/opt/miniconda3/envs/qwen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = 'Qwen/Qwen3-0.6B'
lora_path = "models/qwen-sst2-lora"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, trust_remote_code=True, device_map="mps")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = PeftModel.from_pretrained(model, lora_path)

Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Qwen3ForSequenceClassification(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
          

In [ ]:
for name, param in model.named_parameters():
        if 'lora' in name:
            param.requires_grad = True
def extract_lora_params(model):
    lora_params = {}
    for name, param in model.named_parameters():
        if param.requires_grad:
            lora_params[name] = param.data
    return lora_params

lora_params = extract_lora_params(model)
print(lora_params)

In [8]:
for name, param in lora_params.items():
    print(name)
    print(param.shape)

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
torch.Size([8, 1024])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
torch.Size([2048, 8])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
torch.Size([8, 1024])
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
torch.Size([1024, 8])
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
torch.Size([8, 1024])
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight
torch.Size([2048, 8])
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight
torch.Size([8, 1024])
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight
torch.Size([1024, 8])
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight
torch.Size([8, 1024])
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight
torch.Size([2048, 8])
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight